# New must glycerol estimability and optimal experimental design

This notebook documents the calibration-prior, estimability, and model-based design workflow for the new natural and synthetic must datasets. CO2 and aroma states are intentionally excluded in this iteration; dead biomass and glycerol are included.

## Model extension

The state vector is

$$z(t)=\left[X, X_d, N, G, F, E, Gly\right]^T.$$ 

The base kinetic equations retain the effective Zenteno formulation used in the previous notebook. Dead biomass is represented as

$$\frac{dX_d}{dt}=K_d(T,E)X,$$

and viable biomass as

$$\frac{dX}{dt}=\left(\mu-K_d\right)X + u_X(t).$$

Glycerol is added as a fermentation-associated by-product:

$$\frac{dGly}{dt}=\left(\gamma_{G,0}\,\phi_G(T,G,E)+\gamma_{F,0}\,\phi_F(T,F,G,E)\right)X.$$

The factors $\phi_G$ and $\phi_F$ are the same effective fermentation factors that multiply the ethanol rates. This makes glycerol informative for the same sugar/ethanol-inhibition subspace, rather than treating it as an independent empirical curve.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT / 'fermentation_model').exists():
    ROOT = ROOT / 'fermentation_model'
RESULTS = ROOT / 'results/new_must_glycerol_estimability_doe'
fit_summary = pd.read_csv(RESULTS / 'fit_summary.csv')
param_summary = pd.read_csv(RESULTS / 'parameter_estimability_summary.csv')
candidate_ranking = pd.read_csv(RESULTS / 'candidate_ranking.csv')
selected_hybrid = pd.read_csv(RESULTS / 'selected_campaign_hybrid.csv')
selected_dopt = pd.read_csv(RESULTS / 'selected_campaign_d_opt.csv')
fit_summary

## Calibration prior

Three calibration views are fitted: natural-only, synthetic-only, and mixed. The practical prior used for DOE is the mixed full-parameter L2 fit, because it preserves the complete parameter covariance while avoiding uncontrolled drift of weak directions.

In [ ]:
display(fit_summary.sort_values('final_wsse'))
display(param_summary.sort_values(['analysis', 'std_log_approx']).head(30))

## Medium comparison

A parameter is considered reliable only if it is locally informed by the FIM and does not sit on an active bound. Medium-specific differences should be interpreted as structural/matrix-transfer evidence, not only as noise: natural and synthetic musts occupy different sugar and matrix regimes.

In [ ]:
pivot = param_summary.pivot_table(index='parameter', columns='analysis', values='std_log_approx', aggfunc='first')
pivot.sort_index()

## Profile likelihood

The profile likelihood check refits nuisance parameters while fixing selected parameters over a bounded local grid. A parameter is practically identifiable at 95% confidence when the likelihood-ratio curve crosses the $\chi^2_1(0.95)$ threshold on both sides of the estimate.

In [ ]:
profile_path = RESULTS / 'profile_summary_combined.csv'
if not profile_path.exists():
    profile_path = RESULTS / 'profile_summary.csv'
pd.read_csv(profile_path) if profile_path.exists() else 'profile summary not available'

## Model-based design of experiments

Candidate experiments are evaluated by their expected Fisher information matrix. The current-data FIM is used as prior information, and each new candidate contributes an additive FIM under the local linear approximation.

The main objective used for the proposed campaign is a hybrid deterministic score:

$$\Psi = \log\det(F) + 2\log\left(\frac{\lambda_{min}(F)}{\lambda_{max}(F)}\right) - 0.1\log\operatorname{tr}(F^{-1}).$$

This keeps D-optimality as the main information-volume criterion while penalizing designs that leave a very weak eigen-direction. A pure D-optimal benchmark is also reported.

In [ ]:
display(candidate_ranking.head(12))
display(selected_hybrid)
display(selected_dopt)

## Input plots

The generated PNG files in `results/new_must_glycerol_estimability_doe/plots` show temperature setpoints, initial states, pulse timings, and feasible sampling windows for each selected candidate.

In [ ]:
from IPython.display import Image, display
for png in sorted((RESULTS / 'plots').glob('campaign_inputs_*.png'))[:5]:
    print(png.name)
    display(Image(filename=str(png)))

## Pyomo.DoE usage

The runner includes a discrete Pyomo model with the same seven-state kinetics and labels `unknown_parameters`, `experiment_outputs`, and `measurement_error` for `pyomo.contrib.doe.DesignOfExperiments`. The saved `pyomo_doe_check.csv` records the sequential FIM check for the first selected experiment when Ipopt is available.

In [ ]:
pyomo_path = RESULTS / 'pyomo_doe_check.csv'
pd.read_csv(pyomo_path) if pyomo_path.exists() else 'Pyomo.DoE check not available'

## How to interpret the output

Use the calibration tables to decide which parameters are already estimable from current data. Use the campaign table to decide which new fermentations most reduce weak parameter variances and improve the smallest FIM eigenvalues. After the first block of three fermentations, rerun this notebook using those data as prior information before executing the next block.